# Code explanation Assistant

## Provides detailed explanation of the pasted/file code


In [1]:
# imports

import os
import io
import sys
from dotenv import load_dotenv
from openai import OpenAI
import gradio as gr
from IPython.display import Markdown, display

In [2]:
load_dotenv(override=True)
openai_api_key = os.getenv('OPENAI_API_KEY')
anthropic_api_key = os.getenv('ANTHROPIC_API_KEY')
google_api_key = os.getenv('GOOGLE_API_KEY')

if openai_api_key:
    print(f"OpenAI API Key exists and begins {openai_api_key[:8]}")
else:
    print("OpenAI API Key not set")

if anthropic_api_key:
    print(f"Anthropic API Key exists and begins {anthropic_api_key[:7]}")
else:
    print("Anthropic API Key not set (and this is optional)")    
    
if google_api_key:
    print(f"Google API Key exists and begins {google_api_key[:2]}")
else:
    print("Google API Key not set (and this is optional)")

OpenAI API Key exists and begins sk-proj-
Anthropic API Key exists and begins sk-ant-
Google API Key exists and begins AI


In [3]:
# Connect to client libraries

openai = OpenAI()

anthropic_url = "https://api.anthropic.com/v1/"
gemini_url = "https://generativelanguage.googleapis.com/v1beta/openai/"
ollama_url = "http://localhost:11434/v1"

anthropic = OpenAI(api_key=anthropic_api_key, base_url=anthropic_url)
gemini = OpenAI(api_key=google_api_key, base_url=gemini_url)
ollama = OpenAI(api_key="ollama", base_url=ollama_url)

In [4]:
models = ["gpt-5", "claude-sonnet-4-5-20250929", "gemini-2.5-pro", "qwen2.5-coder", "deepseek-coder-v2"]

clients = {"gpt-5": openai, "claude-sonnet-4-5-20250929": anthropic, "gemini-2.5-pro": gemini, "qwen2.5-coder": ollama, "deepseek-coder-v2": ollama}

In [5]:
system_message = f"""
You are an expert programming tutor and code explanation assistant.

Your task is to explain code shared by the user in a clear, structured, and educational way.

For every code snippet:
1. Provide a one-paragraph overview of what the code does.
2. Explain the code block by block or line by line, depending on complexity.
3. Describe the purpose of key variables, functions, classes, libraries, and logic.
4. Explain the flow of execution from start to finish.
5. Highlight important programming/SQL concepts used in the code.
6. Identify any bugs, inefficiencies, or edge cases.
7. Suggest improvements or cleaner alternatives when relevant.
8. Include an example input/output or real-world analogy if it helps understanding.

Adapt your explanation to the user's level:
- Beginner: use simple language and avoid jargon.
- Intermediate: explain logic and code structure clearly.
- Advanced: include technical details, trade-offs, and deeper reasoning.

Be accurate, helpful, and easy to follow. If the code is ambiguous or incomplete, state that clearly instead of guessing.
"""

In [6]:
LANGUAGE_HINTS = {
    "Auto Detect": "Detect the programming language from the code and explain accordingly.",
    "Python": "The code is written in Python.",
    "SQL": "The code is written in SQL.",
    "PySpark": "The code is written in PySpark.",
    "Java": "The code is written in Java.",
    "JavaScript": "The code is written in JavaScript.",
    "TypeScript": "The code is written in TypeScript.",
    "Scala": "The code is written in Scala.",
    "C++": "The code is written in C++.",
    "R": "The code is written in R.",
    "Other": "The code may be in another programming language. Detect it correctly."
}

In [ ]:
def build_user_prompt(code: str, language: str) -> str:
    language_hint = LANGUAGE_HINTS.get(
        language,
        "Detect the programming language from the code and explain accordingly."
    )

    return f"""
{language_hint}

Please explain the following code in markdown format.

Code:
```text
{code}"""

In [10]:
build_user_prompt("def","SQL")

'\nThe code is written in SQL.\n\nPlease explain the following code in markdown format.\n\nCode:\n```text\ndef'

In [11]:
def read_uploaded_file(file_obj):
    if file_obj is None:
        return ""
    try:
        with open(file_obj.name, "r", encoding="utf-8") as f:
            return f.read()
    except UnicodeDecodeError:
        with open(file_obj.name, "r", encoding="latin-1") as f:
            return f.read()
    except Exception as e:
            return f"Error reading file: {str(e)}"

def load_file_to_textbox(file_obj):
    return read_uploaded_file(file_obj)

In [12]:
def get_code_explanation(code_text, uploaded_file, selected_model, selected_language):
    code_text = (code_text or "").strip()

    if not code_text and uploaded_file is not None:
        code_text = read_uploaded_file(uploaded_file).strip()

    if not code_text:
        yield "⚠️ Please paste some code or upload a code file."
        return

    if selected_model not in clients:
        yield f"⚠️ Model '{selected_model}' is not configured in clients."
        return
    
    try:
        client = clients[selected_model]
        user_prompt = build_user_prompt(code_text, selected_language)   

        stream = client.chat.completions.create(
        model=selected_model,
        messages=[
            {"role": "system", "content": system_message},
            {"role": "user", "content": user_prompt}
        ],
        stream=True
        )

        full_response = ""
        for chunk in stream:
            if not chunk.choices:
                continue

            delta = chunk.choices[0].delta
            content = getattr(delta, "content", None)

            if content:
                full_response += content
                yield full_response

        if not full_response.strip():
            yield "⚠️ No response was returned by the model."

    except Exception as e:
        yield f"❌ Error\n\n```text\n{str(e)}\n```"

In [13]:
def clear_all():
    return "", None, models[0], "Auto Detect", ""

In [16]:
force_dark_mode = """
function refresh() {
    const url = new URL(window.location);
    if (url.searchParams.get('__theme') !== 'dark') {
        url.searchParams.set('__theme', 'dark');
        window.location.href = url.href;
    }
}
"""

In [ ]:
with gr.Blocks(title="Code Explanation Assistant", js=force_dark_mode) as demo:
    gr.Markdown(""" Code Explanation Assistant

        Paste code or upload a file, choose a model and language, and get a streaming markdown explanation.

        Supported examples:

        Python
        SQL
        PySpark
        Java
        JavaScript
        TypeScript
        Scala
        C++
        R
        """
    )
    with gr.Row():
        # Left side
        with gr.Column(scale=2):
            model_input = gr.Dropdown(
                choices=models,
                value=models[0],
                label="Select Model"
            )

            language_input = gr.Dropdown(
                choices=[
                    "Auto Detect",
                    "Python",
                    "SQL",
                    "PySpark",
                    "Java",
                    "JavaScript",
                    "TypeScript",
                    "Scala",
                    "C++",
                    "R",
                    "Other"
                ],
                value="Auto Detect",
                label="Select Language"
            )

            code_input = gr.Textbox(
                label="Paste Code",
                lines=10,
                placeholder="Paste your code here. If both pasted code and file are provided, pasted code will be used first."
            )

            file_input = gr.File(
                label="Upload Code File",
                file_types=[".py", ".sql", ".java", ".js", ".ts", ".scala", ".cpp", ".r", ".txt"]
            )

            file_input.change(
                fn=load_file_to_textbox,
                inputs=file_input,
                outputs=code_input
            )

            with gr.Row():
                explain_btn = gr.Button("Explain Code", variant="primary")
                clear_btn = gr.Button("Clear")

        # Right side
        with gr.Column(scale=2):
            output_md = gr.Markdown(
                value="_Explanation will appear here after you click **Explain Code**._",
                label="Explanation",
                padding=True,
                container=True,
                min_height=500
            )

    explain_btn.click(
        fn=get_code_explanation,
        inputs=[code_input, file_input, model_input, language_input],
        outputs=output_md
    )

    clear_btn.click(
        fn=clear_all,
        inputs=[],
        outputs=[code_input, file_input, model_input, language_input, output_md]
    )


TypeError: Markdown.__init__() got an unexpected keyword argument 'placeholder'

In [27]:
#demo.queue()
demo.launch(inbrowser=True,share=True)

* Running on local URL:  http://127.0.0.1:7864

Could not create share link. Please check your internet connection or our status page: https://status.gradio.app.
